In [37]:
using TSSOS
using LinearAlgebra
using QuantumOptics
using DynamicPolynomials
using QuadGK
using JuMP
using Random
using NLopt

## Define the quantum system

In [38]:
# We look at a typical Hamiltonian

#Drift Hamiltonian

H0 = [
    0 0 0;
    0 0.515916 0;
    0 0 1
];

H0 ./= norm(H0, Inf)

# Control Hamiltonian
V = [
    0 0.707107 0;
    0.707107 0 1;
    0 1 0
]

V ./= norm(V, Inf);

In [39]:
H0*V-V*H0

3×3 Matrix{Float64}:
 0.0       -0.364808   0.0
 0.364808   0.0       -0.484084
 0.0        0.484084   0.0

## Useful functions with polynomials

In [40]:
function ∫(p::AbstractPolynomial, x::PolyVar, x_lower, x_upper)
    
    # get the index of the variable of integration
    ind_x = indexin([x], variables(p))[1]
        
    if isnothing(ind_x)
        # integration valuable is not found among vars
        return p * (x_upper - x_lower)
    end
    
    # get the indefinite integral
    int_p = sum(
        term * x * 1 // (exponents(term)[ind_x] + 1) for term in terms(p)
        init = 0 * x
    )
            
    # get the definite integral
    subs(int_p, x=>x_upper) - subs(int_p, x=>x_lower)
end

function ∫(M::AbstractMatrix, x::PolyVar, x_lower, x_upper)
   map(z -> ∫(z, x, x_lower, x_upper), M) 
end

function real_poly(p::Polynomial)
    #=
    Real part of the polynomial
    =#
    sum(
        real(c) * m for (c, m) in zip(coefficients(p), monomials(p))# if ~isapproxzero(abs(c))
    )
end

function square_frobenius_norm(M::AbstractArray)
    #=
    Square of the Frobenius norm of a matrix
    =#
    real_poly(sum(z' * z for z in M))
end

function square_frobenius_norm2(M::AbstractArray)
    #=
    Square of the Frobenius norm of a matrix
    =#
    sum(z' * z for z in M)
end

function trace_matrix(M::AbstractMatrix)
    d = size(M, 1)
    return sum(M[i, i] for i in 1:d)
end

function traceless_projection(M::AbstractMatrix)
    d = size(M, 1)
    trM = trace_matrix(M)
    return [M[i, j] - (i == j ? trM / d : zero(M[i, j])) for i in 1:d, j in 1:d]
end

traceless_projection (generic function with 1 method)

## Magnus expansion

In [41]:
@polyvar x[1:3]

# final time
const T = 0.5

function u(t, x)
    # Fourier control: constant plus first cosine/sine harmonic.
    x[1] + x[2] * cos(2π * t / T) + x[3] * sin(2π * t / T)
end

function A(t, x,)
    #=
    The generator of motion entering the Magnus expansion
    =#
    (H0 + V * u(t, x)) / im
end

function commutator(a, b)
    a * b - b * a
end 

function gausslegendre(n::Integer)
    beta = [k / sqrt(4k^2 - 1) for k in 1:(n - 1)]
    F = eigen(SymTridiagonal(zeros(n), beta))
    F.values, 2 .* F.vectors[1, :].^2
end

function integrate_time(f, a, b; n=16)
    nodes, weights = gausslegendre(n)
    mid = (a + b) / 2
    half_width = (b - a) / 2
    acc = zero(f(mid))
    for k in eachindex(nodes)
        acc .+= half_width * weights[k] .* f(mid + half_width * nodes[k])
    end
    acc
end

function magnus_expansion(x; quad_order=16)
    Ω = integrate_time(t1 -> A(t1, x), 0, T; n=quad_order)

    Ω .+= 1//2 * integrate_time(t1 -> integrate_time(
        t2 -> commutator(A(t1, x), A(t2, x)),
        0, t1; n=quad_order),
        0, T; n=quad_order
    )

    Ω .+= 1//6 * integrate_time(t1 -> integrate_time(t2 -> integrate_time(
        t3 -> commutator(A(t1, x), commutator(A(t2, x), A(t3, x))) +
              commutator(commutator(A(t1, x), A(t2, x)), A(t3, x)),
        0, t2; n=quad_order),
        0, t1; n=quad_order),
        0, T; n=quad_order
    )

    Ω
end

# get the partial sum of the Magnus expansion
Ω = magnus_expansion(x);


## Optimize fixed three-level target gates

In [42]:
function get_unitary(x::AbstractArray)
    #=
    Get the unitary given the coefficients for the Fourier control
    =#
    basis = NLevelBasis(size(H0)[1])

    𝓗₀ = DenseOperator(basis, basis, H0)
    𝓥 = DenseOperator(basis, basis, V)

    H = LazySum([1., u(0, x)], [𝓗₀, 𝓥])
        
    function 𝓗(t, psi)
        H.factors[2] = u(t, x)
        return H
    end

    _, 𝓤 = timeevolution.schroedinger_dynamic([0, T], identityoperator(basis, basis), 𝓗)
    
    return Matrix(𝓤[2].data)
end

get_unitary (generic function with 1 method)

In [43]:
function logarithm_mat(U)
    Uc = ComplexF64.(U)
    F = eigen(Uc)
    X = F.vectors
    n = 0
    theta_values = [log(complex(λ)) for λ in F.values]
    K = Diagonal(theta_values .+ 2π * im * n)
    X * K * inv(X)
end

logarithm_mat (generic function with 1 method)

## Functions to compare

In [44]:
function Magnus(x)
    magnus_expansion(x)
end

Magnus (generic function with 1 method)

In [45]:
function local_minimize(obj::AbstractPolynomial, init_x::AbstractArray)
    #=
    Perform local minimization of obj polynomial using init_x as initial guess
    =#
    vars = variables(obj)

    @assert length(vars) == length(init_x)
    
    function g(a...)
        # Converting polynomial expression to function to be minimize
        obj(vars => a)
    end
    
    model = Model(NLopt.Optimizer)

    set_optimizer_attribute(model, "algorithm", :LD_MMA)

    set_silent(model)
    @variable(model, y[1:length(vars)])

    # set initial guess
    for (var, init_val) in zip(y, init_x)
        set_start_value(var, init_val)
    end

    register(model, :g, length(y), g; autodiff = true)
    @NLobjective(model, Min, g(y...))
    JuMP.optimize!(model)

    map(value, y)
end

local_minimize (generic function with 1 method)

## TSSOS

In [46]:
@time begin
    
function rotation_gate(i::Int, j::Int, theta)
    U = Matrix{ComplexF64}(I, 3, 3)
    c = cos(theta)
    s = sin(theta)
    U[i, i] = c
    U[j, j] = c
    U[i, j] = -s
    U[j, i] = s
    U
end

function fourier_gate(d::Int)
    ω = exp(2π * im / d)
    [ω^((row - 1) * (col - 1)) / sqrt(d) for row in 1:d, col in 1:d]
end

function phase_rotation_gate(i::Int, j::Int, theta, phi)
    U = Matrix{ComplexF64}(I, 3, 3)
    c = cos(theta)
    s = sin(theta)
    U[i, i] = c
    U[j, j] = c
    U[i, j] = -exp(-im * phi) * s
    U[j, i] = exp(im * phi) * s
    U
end

function permutation_gate(order)
    U = zeros(ComplexF64, 3, 3)
    for col in 1:3
        U[order[col], col] = 1
    end
    U
end

nan_control = fill(NaN, length(x))
target_catalog_names = String[]
target_catalog_controls = Vector{Vector{Float64}}()
target_catalog_unitaries = Matrix{ComplexF64}[]

function add_target!(name, U; planted_x=nan_control)
    push!(target_catalog_names, name)
    push!(target_catalog_controls, collect(Float64, planted_x))
    push!(target_catalog_unitaries, Matrix{ComplexF64}(U))
end

# Structured gates, from gentle local rotations to harder global transforms.
add_target!("identity", Matrix{ComplexF64}(I, 3, 3))
add_target!("rotation_01_pi8", rotation_gate(1, 2, π / 8))
add_target!("rotation_01_pi4", rotation_gate(1, 2, π / 4))
add_target!("rotation_01_pi2", rotation_gate(1, 2, π / 2))
add_target!("rotation_12_pi8", rotation_gate(2, 3, π / 8))
add_target!("rotation_12_pi4", rotation_gate(2, 3, π / 4))
add_target!("rotation_12_pi2", rotation_gate(2, 3, π / 2))
add_target!("rotation_02_pi4", rotation_gate(1, 3, π / 4))
add_target!("phase_rotation_01_pi4", phase_rotation_gate(1, 2, π / 4, π / 2))
add_target!("phase_rotation_12_pi4", phase_rotation_gate(2, 3, π / 4, π / 2))
add_target!("diagonal_phase_small", Matrix{ComplexF64}(Diagonal([1, exp(im * π / 8), exp(-im * π / 8)])))
add_target!("diagonal_phase_medium", Matrix{ComplexF64}(Diagonal([1, exp(im * π / 3), exp(-im * π / 5)])))
add_target!("diagonal_phase_large", Matrix{ComplexF64}(Diagonal([1, exp(im * π / 2), exp(im * π)])))
add_target!("swap_01", permutation_gate([2, 1, 3]))
add_target!("swap_12", permutation_gate([1, 3, 2]))
add_target!("cycle_012", permutation_gate([2, 3, 1]))
add_target!("cycle_021", permutation_gate([3, 1, 2]))
add_target!("fourier_dft", fourier_gate(3))
add_target!("fourier_dft_dagger", fourier_gate(3)')

# Hamiltonian-native or near-native gates that should be easier or reveal reachability structure.
add_target!("drift_short", exp(H0 / im * (T / 2)))
add_target!("drift_free", get_unitary([0.0, 0.0, 0.0]), planted_x=[0.0, 0.0, 0.0])
add_target!("drift_long", exp(H0 / im * (2T)))
add_target!("control_only_small", exp((0.25 * V) / im * T))
add_target!("control_only_large", exp((0.75 * V) / im * T))
add_target!("const_drive_p20", get_unitary([0.20, 0.0, 0.0]), planted_x=[0.20, 0.0, 0.0])
add_target!("const_drive_p50", get_unitary([0.50, 0.0, 0.0]), planted_x=[0.50, 0.0, 0.0])
add_target!("const_drive_m50", get_unitary([-0.50, 0.0, 0.0]), planted_x=[-0.50, 0.0, 0.0])
add_target!("cos_drive_p35", get_unitary([0.0, 0.35, 0.0]), planted_x=[0.0, 0.35, 0.0])
add_target!("sin_drive_p35", get_unitary([0.0, 0.0, 0.35]), planted_x=[0.0, 0.0, 0.35])
add_target!("mixed_fourier_soft", get_unitary([0.15, -0.30, 0.20]), planted_x=[0.15, -0.30, 0.20])
add_target!("mixed_fourier_medium", get_unitary([0.35, 0.25, -0.30]), planted_x=[0.35, 0.25, -0.30])
add_target!("mixed_fourier_strong", get_unitary([0.60, -0.40, 0.35]), planted_x=[0.60, -0.40, 0.35])
add_target!("oscillatory_balanced", get_unitary([0.0, 0.65, -0.65]), planted_x=[0.0, 0.65, -0.65])

# Edit this list to choose which catalog targets to run.
selected_target_names = copy(target_catalog_names)
selected_target_indices = Int[]
for name in selected_target_names
    idx = findfirst(==(name), target_catalog_names)
    @assert !isnothing(idx) "selected_target_names contains an unknown target: $(name)"
    push!(selected_target_indices, idx)
end

target_names = selected_target_names
target_unitaries = target_catalog_unitaries[selected_target_indices]
target_planted_x = reduce(hcat, target_catalog_controls[selected_target_indices])

n_samples = length(target_unitaries)

# Baseline controls used only to report the objective before optimization.
initial_x = zeros(length(x), n_samples)
obj_initial_x = zeros(n_samples)
    
# The fixed target unitaries.
U_targets = zeros(ComplexF64, n_samples, size(H0)...)

# Optimized Fourier coefficients.
opt_x = zeros(length(x), n_samples)

# the polynomial objective at min_x
glob_obj_min_x = zeros(n_samples)

# The global minimum via TSSOS library
tssos_glob_obj_min = zeros(n_samples)

# Frobenius norm difference between target and obtained unitaries
norm_U_target_minus_obtained = zeros(n_samples)

# Frobenius norm difference between target and the truncated Magnus expansion
norm_U_target_minus_expΩ_initial_x = zeros(n_samples)
norm_U_target_minus_expΩ_min_x = zeros(n_samples)

# The normalised overlap of the evolution and the target 
f_PSU = zeros(n_samples) 

# Convergence test for the Magnus expansion for the baseline control (convergence_test_initial_x < 1)
convergence_test_initial_x = zeros(n_samples)


# Convergence test for the Magnus expansion for the obtained control (convergence_test_min_x < 1)
convergence_test_min_x = zeros(n_samples)

# Per-target POP runtime, including objective construction, TSSOS solve, and diagnostics.
pop_runtime = zeros(n_samples)

Threads.@threads for i=1:n_samples
    target_start = time()
    
    # target unitary
    U_targets[i, :, :] = U_target = target_unitaries[i]
        
    # get the polynomial objective function
    Theta = logarithm_mat(U_targets[i,:,:])

    # Ω is the anti-Hermitian truncated Magnus logarithm Λ_n.
    # Convert to Hermitian generators: G = iΛ.
    G_n = im * Ω
    G_star = im * Theta

    obj = square_frobenius_norm(
        traceless_projection(G_n - G_star)
    )
    
    # save the value of objective function for the baseline control
    obj_initial_x[i] = obj(initial_x[:, i])
  
    
    # Get the global minimum via TSSOS library
    opt,sol,data = tssos_first(obj, variables(obj); QUIET = true, solution = true)
    
    previous_sol = sol
    previous_opt = opt
    
    while ~isnothing(sol)
        previous_sol = sol
        previous_opt = opt
            
        opt,sol,data = tssos_higher!(data; QUIET = true, solution = true)
    end
    tssos_glob_obj_min[i] = previous_opt
    min_x = previous_sol
    opt_x[:, i] = min_x
    
    # refine the estimate by local minimization
    #min_x = local_minimize(obj, min_x)
    
    # the polynomial objective at min_x
    glob_obj_min_x[i] = tssos_glob_obj_min[i]
        
    # get the Frobenius norm difference between target and obtained unitaries
    U_star = get_unitary(min_x)
    norm_U_target_minus_obtained[i] = norm(U_target - U_star)

    # Projective/process fidelity: |Tr(U_target' * U_star)|^2 / d^2
    d = size(U_star, 1)
    f_PSU[i] = (real(abs(tr(U_target' * U_star))/ d)^2)
        
    # check the accuracy of the Magnus expansion
    Ω_initial_x = convert(Matrix{ComplexF64}, subs(Ω, x=>initial_x[:, i]))
    norm_U_target_minus_expΩ_initial_x[i] = norm(U_target - exp(Ω_initial_x))

    Ω_min_x = convert(Matrix{ComplexF64}, subs(Ω, x=>min_x))
    norm_U_target_minus_expΩ_min_x[i] = norm(U_target - exp(Ω_min_x))
    
    # Convergence test for the Magnus expansion for the baseline control
    convergence_test_initial_x[i] = quadgk(t -> opnorm(A(t, initial_x[:, i])), 0, T)[1] / π

    # Convergence test for the Magnus expansion for the obtained control
    convergence_test_min_x[i] = quadgk(t -> opnorm(A(t, min_x)), 0, T)[1] / π    

    pop_runtime[i] = time() - target_start
end

pop_average_runtime = sum(pop_runtime) / length(pop_runtime)

end

*********************************** TSSOS ***********************************
TSSOS is launching...
optimum = 0.029236953861921103
Global optimality certified with relative optimality gap 0.000005%!
No higher TS step of the TSSOS hierarchy!
*********************************** TSSOS ***********************************
TSSOS is launching...
optimum = 0.22594480953041968
Found a locally optimal solution by Ipopt, giving an upper bound: 0.22634555.
The relative optimality gap is: 0.040074%.
No higher TS step of the TSSOS hierarchy!
*********************************** TSSOS ***********************************
TSSOS is launching...
optimum = 0.8917017830057876
Global optimality certified with relative optimality gap 0.000011%!
No higher TS step of the TSSOS hierarchy!
*********************************** TSSOS ***********************************
TSSOS is launching...
optimum = 3.9631613285460476
Global optimality certified with relative optimality gap 0.000009%!
No higher TS step of the TSSOS

0.09642424005450624

In [47]:
using HDF5


h5open("results.hdf5", "w") do fid
    fid["target_names"] = join(target_names, "\n")
    fid["target_catalog_names"] = join(target_catalog_names, "\n")
    fid["U_targets"] = U_targets
    fid["target_planted_x"] = target_planted_x
    fid["initial_x"] = initial_x
    fid["opt_x"] = opt_x
    fid["obj_initial_x"] = obj_initial_x
    fid["tssos_glob_obj_min"] = tssos_glob_obj_min
    fid["glob_obj_min_x"] = glob_obj_min_x
    fid["norm_U_target_minus_obtained"] = norm_U_target_minus_obtained
    fid["norm_U_target_minus_expΩ_initial_x"] = norm_U_target_minus_expΩ_initial_x
    fid["norm_U_target_minus_expΩ_min_x"] = norm_U_target_minus_expΩ_min_x
    fid["f_PSU"] = f_PSU
    fid["convergence_test_initial_x"] = convergence_test_initial_x
    fid["convergence_test_min_x"] = convergence_test_min_x
    fid["pop_runtime"] = pop_runtime
    fid["pop_average_runtime"] = pop_average_runtime
end;